In [27]:
from ask_delphi_api.authentication import AskDelphiClient
from ask_delphi_api.workflow import Workflow
from ask_delphi_api.import_digicoach import Import

client = AskDelphiClient()
client.authenticate()

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 401
Cached tokens failed: Failed to fetch editing API token:

Exchanging portal code...
Status: 200
Access token received.
Publication URL: https://digitalecoach.askdelphi.com
Tokens saved to cache.
Getting editing API token...
Editing API token status: 200
Editing API token acquired.
AUTHENTICATION SUCCESSFUL


True

In dit notebook wordt uitgewerkt hoe je topics (op basis van hun timestamp) kunt verwijderen.
Stappenplan
- Ophalen van alle info van alle bestaande topics
- topics filteren op basis van LastModificationDate
- topics die voldoen uitchecken en meteen verwijderen

In [28]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools

project = Project(client)
topics = TopicTools(client, project)

import_service = Import()

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


In [29]:
# all_topics = topics.fetch_topiclist()

In [30]:
import pprint
# pprint.pprint(all_topics[1001])

# "b0e911e4-3d0c-432b-bb6c-d06ac939a037"

In [31]:

# def get_workflowstate(topicId: str):  
#     endpoint = f"/v3/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/workflowstate"
#     data = {}
#     result = client._request("GET", endpoint, json_data=data)
#     return result

# pprint.pprint(get_workflowstate("b0e911e4-3d0c-432b-bb6c-d06ac939a037"))

In [32]:
# results = topics.filter_between("2026-01-13T00:00:00", "2026-02-18T23:59:59")
# print("Aantal topics binnen de range:", len(results))

In [33]:
# topicIdverwijderen = [r["topicGuid"] for r in results]

# for topic_id in topicIdverwijderen:
#     try: 
#         version_id = topics.get_topicVersionId(topic_id)
#         result = topics.delete_topic(topic_id, version_id)
#     except Exception as e:
#         print(f"FOUT bij verwijderen van {topic_id}: {e}")



In [34]:

endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/workflow/search"
data = {}
response = client._request("POST", endpoint, json_data=data)

# print(response)

In [35]:
endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/workflow/{'d5dc5516-a8c7-43a9-bb59-c73dd1d0682b'}"
data = {}
response = client._request("GET", endpoint, json_data=data)

# pprint.pprint(response)

In [ ]:
def get_topic_relation(topicId: str):
    """Opvragen topic relations."""
    endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/relation"
    data = {}
    return client._request("GET", endpoint, json_data=data)

def delete_topic_relation(sourceTopicId: str, targetTopicId: str, relationTypeId: str):
    sourceTopicVersionId = topics.get_topicVersionId(sourceTopicId)
    """Verwijder een topic relatie."""
    endpoint = f"v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{sourceTopicId}/topicVersion/{sourceTopicVersionId}/relation/delete"
    data = {
        "relationTypeId": relationTypeId,
        "sourceTopicId": sourceTopicId,
        "targetTopicId": targetTopicId
    }
    return client._request("POST", endpoint, json_data=data)

def delete_topic(topicId: str, topicVersionId: str):
    """Verwijder een topic."""
    endpoint = f"v3/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/topicVersion/{topicVersionId}"
    data = {
        "workflowActions": {
            "increaseMajorVersionNo": True
        }
    }
    return client._request("DELETE", endpoint, json_data=data)


### Opvragen topic relaties

In [ ]:
topic_id_digicoach = "3cbeb7f4-808b-41e9-af98-f8a99110355d"
topic_id_taak = "2fc54097-0251-47f8-a042-b9c51b9353a0"
topic_id_stap = "632f4f62-3112-435f-b780-04020b8cbda0"

### Verwijder topic relatie Taak -> Stap

In [ ]:
topics.checkout(topic_id_taak)
topic_version_id_taak = topics.get_topicVersionId(topic_id_taak)

relatie_type_id = import_service.relation.get_relation_type_id(topic_id_taak, topic_version_id_taak, "Stap")

response = delete_topic_relation(topic_id_taak, topic_id_stap, relatie_type_id)

response = topics.checkin(topic_id_taak)
import_service.publiceer(topic_id_taak)

relatie_type_id : 85df8887-1ead-40e7-8c1a-499aa8621d00
2026-03-05T17:27:59Z


### Verwijder topic

In [ ]:
topics.checkout(topic_id_stap)
topic_version_id_stap = topics.get_topicVersionId(topic_id_stap)

try:
    response = delete_topic(topic_id_stap, topic_version_id_stap) 
except:
    print("Oeps markeren opruimen lijkt mislukt")